# M8 vehicle detector, step 18: score the newly trained models on the full UVH-26 validation split

Runs the new candidate model(s) over **all 5,297 labelled validation frames**, together with the repaired IISc model (`iisc_bnfix_t`, the same file as before) so that the
speed target compares two models measured in the same run on the same machine. The judging (P1 to P5, locked in `baselines/2026-10-28_m8_full_train/TARGETS_LOCKED.md`) is done afterwards on the returned detections.

**Settings:** Accelerator **None** (CPU), **Internet on** (frames come from huggingface.co). Input: the `uvh26_score_kaggle/bundle` folder as a dataset (code, labels, models).
Set **CAND** below, then **Save Version -> Save & Run All**. About 1 to 2 hours. Send back the `uvh26_score_<CAND>_results.zip` from the Output tab.

In [ ]:
CAND = ["s_iisc"]      # the trained models to score; later ["s_coco", "m_coco"]

In [ ]:
import glob, os, subprocess, sys, shutil, time, zipfile, json
assert sys.version_info >= (3, 11), f"needs Python 3.11+, have {sys.version}"
hits = glob.glob("/kaggle/input/**/m8_detector_fullval.py", recursive=True) + glob.glob("**/m8_detector_fullval.py", recursive=True)
assert hits, "bundle not found: add the uvh26_score_kaggle/bundle folder as a dataset input"
ROOT = os.path.dirname(os.path.dirname(os.path.abspath(hits[0])))
BASE = "/kaggle/working" if os.path.isdir("/kaggle/working") else os.getcwd()
TAG = "_".join(CAND); WORK = f"{BASE}/score_{TAG}"; os.makedirs(WORK, exist_ok=True)
for c in CAND + ["iisc_bnfix_t"]:
    assert os.path.isfile(f"{ROOT}/models/{c}.onnx"), f"models/{c}.onnx missing from the bundle (found: {os.listdir(ROOT + '/models')})"
print("bundle:", ROOT, "\nwork:", WORK)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "onnxruntime", "opencv-python-headless", "pycocotools", "numpy"], check=True)

def run(args, cwd=ROOT):
    p = subprocess.Popen([sys.executable, *args], cwd=cwd, env=dict(os.environ, PYTHONPATH=ROOT),
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end="")
    code = p.wait()
    assert code == 0, f"exit {code}"

r = subprocess.run(["curl", "-sSI", "-o", "/dev/null", "-w", "%{http_code}", "https://huggingface.co/datasets/iisc-aim/UVH-26/resolve/main/UVH-26-Val/UVH-26-MV-Val.json"],
                   capture_output=True, text=True)
print("huggingface.co ->", r.stdout or r.stderr)
assert r.stdout.strip() in ("200", "302", "307"), "huggingface.co not reachable: turn Internet on for this notebook"

In [ ]:
MODELS = []
for c in CAND + ["iisc_bnfix_t"]:
    MODELS += ["--model", f"{c}=models/{c}.onnx", "--model14", c]
DETECT = ["scripts/m8_detector_fullval.py", "detect", "--labels", "data/UVH-26-MV-Val.json", *MODELS,
          "--manifest", "data/train_manifest.json", "--providers", "cpu"]
SCORE = ["scripts/m8_detector_fullval.py", "score", "--labels", "data/UVH-26-MV-Val.json",
         "--old250", "data/val_sample.json", "--fresh250", "data/fresh_names.json"]
# smoke test: 40 frames, own output folder (stops here if the download or a model is broken)
shutil.rmtree(f"{WORK}/smoke", ignore_errors=True)
run([*DETECT, "--limit", "40", "--shard", "20", "--tmp", f"{WORK}/tmp_smoke", "--out", f"{WORK}/smoke"])
run([*SCORE, "--run", f"{WORK}/smoke", "--boot", "50", "--out", f"{WORK}/smoke_res"])

In [ ]:
# the full run. Resumable: finished 250-frame shards are skipped, so after a disconnect just run this cell again.
t0 = time.time()
run([*DETECT, "--shard", "250", "--workers", "6", "--tmp", f"{WORK}/tmp", "--out", f"{WORK}/run"])
print(f"detection finished in {(time.time() - t0) / 60:.0f} min")

In [ ]:
# score (same script as the earlier full-validation run) and package
run([*SCORE, "--run", f"{WORK}/run", "--boot", "2000", "--out", f"{WORK}/res"])
ZIP = f"{BASE}/uvh26_score_{TAG}_results.zip"
with zipfile.ZipFile(ZIP, "w", zipfile.ZIP_DEFLATED) as z:
    for d in ("run", "res"):
        for p in sorted(glob.glob(f"{WORK}/{d}/*")):
            if os.path.isfile(p):
                z.write(p, f"{d}/{os.path.basename(p)}")
print(ZIP, round(os.path.getsize(ZIP) / 1e6), "MB")